In [0]:
# CAMADA SILVER
# Databricks notebook source
# COMMAND ----------
from pyspark.sql.functions import col, to_timestamp, trim, upper, when, current_timestamp, hash, concat_ws, abs
from delta.tables import DeltaTable

# Garante criação do schema Silver se não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp_eng_dados.silver")

# ==============================================================================
# 1. TRATAMENTO DA TABELA FATO EGRESSOS
# ==============================================================================

# 1. Leitura da tabela Bronze
df_bronze_egressos = (
    spark.read.table("mvp_eng_dados.bronze.dataset_egressos_bruto")
    # Remove duplicatas por atributos de negócio (mantém apenas um registro por combinação única)
    .dropDuplicates([
        "id_curso", "dt_ultimo_contato", "uf_residencia", 
        "meses_desde_formacao", "idade_egresso", "nivel_cargo",
        "renda_mensal_estimada", "area_atuacao", "modalidade_graduacao",
        "tipo_empresa", "satisfacao_graduacao_nps", "engajamento_alumni_score",
        "bolsista_graduacao", "potencial_matricula_pos"
    ])
)

# 2. Tratamento e saneamento de dados (regras de qualidade de dados)
df_silver_egressos_clean = (
    df_bronze_egressos
    .filter(col("id_egresso").isNotNull())
    .withColumn("id_egresso", col("id_egresso").cast("int"))
    
    # Padronização de strings (Trim + Uppercase)
    .withColumn("id_curso", col("id_curso").cast("int"))
    .withColumn("dt_ultimo_contato", to_timestamp(col("dt_ultimo_contato")))
    .withColumn("uf_residencia", upper(trim(col("uf_residencia"))))
    .withColumn("nivel_cargo", upper(trim(col("nivel_cargo"))))
    .withColumn("area_atuacao", upper(trim(col("area_atuacao"))))
    .withColumn("modalidade_graduacao", upper(trim(col("modalidade_graduacao"))))
    .withColumn("tipo_empresa", upper(trim(col("tipo_empresa"))))
    
    # Validação e saneamento de valores numéricos e limites
    .withColumn("meses_desde_formacao", when(col("meses_desde_formacao") >= 0, col("meses_desde_formacao").cast("int")).otherwise(None))
    .withColumn("idade_egresso", when((col("idade_egresso") >= 16) & (col("idade_egresso") <= 100), col("idade_egresso").cast("int")).otherwise(None))
    .withColumn("renda_mensal_estimada", when(col("renda_mensal_estimada") >= 0, col("renda_mensal_estimada").cast("decimal(10,2)")).otherwise(None))
    
    # Sanidade de notas/escores
    .withColumn("satisfacao_graduacao_nps", when((col("satisfacao_graduacao_nps") >= 0) & (col("satisfacao_graduacao_nps") <= 10), col("satisfacao_graduacao_nps").cast("int")).otherwise(None))
    .withColumn("engajamento_alumni_score", col("engajamento_alumni_score").cast("int"))
    .withColumn("bolsista_graduacao", col("bolsista_graduacao").cast("int"))
    .withColumn("potencial_matricula_pos", col("potencial_matricula_pos").cast("int"))
    
    # Coluna derivada robusta à caixa da string
    .withColumn("is_empregado", when(upper(trim(col("tipo_empresa"))).isin("DESEMPREGADO"), 0).otherwise(1))
    .withColumn("_data_processamento_silver", current_timestamp())
)

# Sobrescreve completamente a tabela Silver para garantir dados limpos
tabela_destino_egressos = "mvp_eng_dados.silver.dataset_egressos"
df_silver_egressos_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_destino_egressos)

# ==============================================================================
# 2. TRATAMENTO DO DATASET DE CURSOS (NOVO)
# ==============================================================================

# Leitura da tabela Bronze
df_bronze_cursos = spark.read.table("mvp_eng_dados.bronze.dataset_cursos_bruto")

# Tratamento, casting e regras de qualidade de dados
df_silver_cursos_clean = (
    df_bronze_cursos
    # Garante que a chave primária não seja nula
    .filter(col("id_curso").isNotNull())
    .withColumn("id_curso", col("id_curso").cast("int"))
    .withColumn("nome_curso", upper(trim(col("nome_curso"))))
    .withColumn("area_conhecimento", upper(trim(col("area_conhecimento"))))
    .withColumn("duracao_semestres", when(col("duracao_semestres") > 0, col("duracao_semestres").cast("int")).otherwise(None))
    .withColumn("mensalidade_base", when(col("mensalidade_base") >= 0, col("mensalidade_base").cast("decimal(10,2)")).otherwise(None))
    .withColumn("_data_processamento_silver", current_timestamp())
    .dropDuplicates(["id_curso"])
)

# Atualização idempotente via MERGE (Upsert)
tabela_destino_cursos = "mvp_eng_dados.silver.dataset_cursos"
if not spark.catalog.tableExists(tabela_destino_cursos):
    df_silver_cursos_clean.write.format("delta").mode("append").saveAsTable(tabela_destino_cursos)
else:
    delta_target_cursos = DeltaTable.forName(spark, tabela_destino_cursos)
    (
        delta_target_cursos.alias("target")
        .merge(
            df_silver_cursos_clean.alias("source"),
            "target.id_curso = source.id_curso"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )